In [ ]:
!nvidia-smi
!pip install -q -U ultralytics
import ultralytics
print(ultralytics.__version__)

Fri Oct 23 05:51:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os, yaml
from google.colab import drive
drive.mount('/content/drive')

FYP = "/content/drive/MyDrive/FYP"
DATA = "/content/data_v3"
if not os.path.exists(f"{DATA}/data.yaml"):
    !unzip -q -o "{FYP}/merged_subset_v3.zip" -d {DATA}

cfg = yaml.safe_load(open(f"{DATA}/data.yaml"))
cfg["path"] = DATA
yaml.safe_dump(cfg, open(f"{DATA}/data.yaml", "w"), sort_keys=False)
for s in ["train", "valid", "test"]:
    print(s, len(os.listdir(f"{DATA}/images/{s}")), "images")
print(cfg["nc"], "classes")

In [ ]:
from ultralytics import YOLO

RUN = "colab-run6-26n"
PROJECT = f"{FYP}/runs"
last = f"{PROJECT}/{RUN}/weights/last.pt"

if os.path.exists(last):
    print("resuming from", last)
    YOLO(last).train(resume=True)
else:
    YOLO("yolo26n.pt").train(data=f"{DATA}/data.yaml", epochs=40, imgsz=640, batch=32,
                             workers=2, project=PROJECT, name=RUN, exist_ok=True, patience=15)

In [ ]:
import pandas as pd

m = YOLO(f"{PROJECT}/{RUN}/weights/best.pt")
r = m.val(data=f"{DATA}/data.yaml", split="test", imgsz=640, batch=32, plots=True)

rows = []
for k, c in enumerate(r.box.ap_class_index):
    p, rec, ap50, ap = r.box.class_result(k)
    rows.append({"class": m.names[int(c)], "precision": p, "recall": rec, "mAP50": ap50, "mAP50-95": ap})
rows.append({"class": "ALL", "precision": r.box.mp, "recall": r.box.mr, "mAP50": r.box.map50, "mAP50-95": r.box.map})
df = pd.DataFrame(rows).round(3)
df.to_csv(f"{PROJECT}/{RUN}/test_per_class.csv", index=False)
print(df.to_string(index=False))
print("ms per image:", {k: round(v, 1) for k, v in r.speed.items()})